# NOTEBOOK : CHAIN LADDER DEPUIS OCTROI (POST-2021)
## Approche classique sans biais de troncature

## Introduction

### Contexte

Dans le notebook principal, nous avons utilisé l'approche **par horizon résiduel** pour éviter le biais de troncature à gauche (contrats octroyés avant 2021).

Ce notebook propose une **approche comparative classique** :
- **Filtrer uniquement les contrats octroyés après 2021** (pas de troncature)
- **Modéliser depuis l'octroi** : ER en fonction de l'âge du prêt
- **Triangles Chain Ladder standards** par cohorte d'octroi

### Objectif

Construire la courbe ER(t) où t = âge du prêt depuis l'octroi, sur des données non tronquées.

### Comparaison des deux approches

| Critère | Horizon résiduel | Depuis octroi (ce notebook) |
|---------|------------------|------------------------------|
| **Question** | "Proba RA dans les h prochains mois ?" | "ER après t mois de vie ?" |
| **Troncature** | Pas de biais (tous observés 2021+) | Biais éliminé (filtre post-2021) |
| **Données** | Tous les contrats | Seulement post-2021 (moins de données) |
| **Opérationnel** | ✅ Projection ALM | ✅ Comparaison littérature |
| **Académique** | Innovant | Standard actuariat |

## Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
print("Bibliotheques chargees")

Bibliotheques chargees


## Etape 1 : Chargement et filtrage

In [6]:
df = pd.read_csv("Export_ER_Agrege.csv", sep=";")

print(f"Dataset complet : {len(df)} contrats")

# Parser date d'observation
df["DARRET"] = pd.to_datetime(df["DARRET"].astype(str) + "01", format="%Y%m%d", errors="coerce")

# Reconstituer date d'octroi approximative (1 échéance = 1 mois)
df['age_mois'] = df['AGE_PRET']  # Age en échéances ≈ mois
df['date_octroi_approx'] =  df["DARRET"] - pd.to_timedelta(df["AGE_PRET"] * 30, unit='D')

# FILTRE : contrats octroyés après 2021-01-01 (pas de troncature)
cutoff_date = pd.Timestamp('2021-01-01')
df_post2021 = df[df['date_octroi_approx'] >= cutoff_date].copy()

print(f"\nContrats post-2021 : {len(df_post2021)} ({len(df_post2021)/len(df):.1%})")
print(f"Periode octroi : {df_post2021['date_octroi_approx'].min()} à {df_post2021['date_octroi_approx'].max()}")
print(f"Taux de RA : {df_post2021['FLAG_ER'].mean():.2%}")

Dataset complet : 63585 contrats

Contrats post-2021 : 0 (0.0%)
Periode octroi : NaT à NaT
Taux de RA : nan%


## Etape 2 : Création des cohortes d'octroi

On regroupe par **trimestre d'octroi**.

In [ ]:
# Cohorte = trimestre d'octroi
df_post2021['Cohorte'] = df_post2021['date_octroi_approx'].dt.to_period('Q')

print("Distribution des cohortes :")
print(df_post2021['Cohorte'].value_counts().sort_index())

print(f"\nNombre de cohortes : {df_post2021['Cohorte'].nunique()}")

## Etape 3 : Construction du triangle agrégé

### Méthodologie Chain Ladder classique

Pour chaque cohorte $c$ et âge $t$ :

$$ER\_cumule(c, t) = \frac{\sum_{i \in c, age=t} RA_i}{\sum_{i \in c} Capital\_initial_i}$$

Le dénominateur (EAD initial) est **fixe par cohorte**.

In [ ]:
# EAD initial par cohorte
ead_initial = df_post2021.groupby('Cohorte')['Capital_initial'].sum().rename('EAD_initial')

print(f"EAD initial total : {ead_initial.sum():,.0f} EUR")
print(f"EAD initial moyen par cohorte : {ead_initial.mean():,.0f} EUR")

# RA agrégé par (cohorte, age)
ra_agrege = (
    df_post2021
    .groupby(['Cohorte', 'age_mois'], as_index=False)['RA'].sum()
    .merge(ead_initial.reset_index(), on='Cohorte')
    .sort_values(['Cohorte', 'age_mois'])
)

# RA cumulé par cohorte
ra_agrege['RA_cumule'] = ra_agrege.groupby('Cohorte')['RA'].cumsum()

# ER cumulé
ra_agrege['ER_cumule'] = ra_agrege['RA_cumule'] / ra_agrege['EAD_initial']
ra_agrege['ER_cumule'] = ra_agrege['ER_cumule'].clip(lower=0)

# Triangle
triangle_cumule = ra_agrege.pivot(index='Cohorte', columns='age_mois', values='ER_cumule').sort_index()

# Volume
triangle_volume = df_post2021.pivot_table(
    index='Cohorte',
    columns='age_mois',
    values='FLAG_ER',
    aggfunc='count'
)

# Filtrer cellules avec < 2 contrats
triangle_filtre = triangle_cumule.copy()
triangle_filtre[triangle_volume < 2] = np.nan

print(f"\nDimensions triangle : {triangle_filtre.shape}")
print(f"Taux remplissage : {(~triangle_filtre.isna()).sum().sum() / triangle_filtre.size:.1%}")

# Afficher extrait
print("\nExtrait triangle (5 cohortes récentes, ages 1-10) :")
print(triangle_filtre.iloc[-5:, :10].round(4))

### Visualisation du triangle

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Heatmap du triangle
sns.heatmap(triangle_filtre.iloc[:, :36], annot=False, cmap='YlOrRd',
            cbar_kws={'label': 'ER cumule'}, ax=axes[0], vmin=0, vmax=1)
axes[0].set_xlabel('Age du pret (mois)', fontsize=12)
axes[0].set_ylabel('Cohorte (trimestre octroi)', fontsize=12)
axes[0].set_title('Triangle cumulé des ER\n(Contrats post-2021 uniquement)', fontsize=13)

# Distribution des ER cumulés observés
er_values = triangle_filtre.values.flatten()
er_values = er_values[~np.isnan(er_values)]
axes[1].hist(er_values, bins=50, edgecolor='black', color='steelblue', alpha=0.7)
axes[1].set_xlabel('ER cumule', fontsize=12)
axes[1].set_ylabel('Frequence', fontsize=12)
axes[1].set_title('Distribution des ER cumules observes', fontsize=13)
axes[1].axvline(np.nanmean(er_values), color='red', linestyle='--', 
                linewidth=2, label=f'Moyenne = {np.nanmean(er_values):.3f}')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"\nER cumule moyen observe : {np.nanmean(er_values):.4f}")
print(f"ER cumule max : {np.nanmax(er_values):.4f}")

## Etape 4 : Calcul des facteurs de développement

$$f_j = \frac{\sum_c ER(c, j+1)}{\sum_c ER(c, j)}$$

In [ ]:
def calculate_cl_factors(triangle, min_cohortes=2):
    factors = {}
    
    for j in range(len(triangle.columns) - 1):
        col_j = triangle.columns[j]
        col_j1 = triangle.columns[j + 1]
        
        mask = triangle[col_j].notna() & triangle[col_j1].notna() & (triangle[col_j] > 0)
        
        if mask.sum() >= min_cohortes:
            sum_j = triangle.loc[mask, col_j].sum()
            sum_j1 = triangle.loc[mask, col_j1].sum()
            
            if sum_j > 0:
                factor = sum_j1 / sum_j
                if 0.8 <= factor <= 3.0:
                    factors[col_j] = factor
                else:
                    factors[col_j] = 1.0
            else:
                factors[col_j] = 1.0
        else:
            factors[col_j] = 1.0
    
    return factors

# Calculer facteurs
factors_cl = calculate_cl_factors(triangle_filtre)

print("FACTEURS DE DEVELOPPEMENT CHAIN LADDER")
print("=" * 70)
print(f"{'Age':>5} {'Facteur':>10} {'Interpretation'}")
print("-" * 70)

for age in sorted(factors_cl.keys())[:24]:
    factor = factors_cl[age]
    if factor >= 1.05:
        interp = "Croissance forte"
    elif factor >= 1.01:
        interp = "Croissance moderee"
    elif factor >= 0.99:
        interp = "Stable"
    else:
        interp = "Decroissance"
    print(f"{age:>5.0f} {factor:>10.4f} {interp}")

# Visualisation
ages_plot = sorted([k for k in factors_cl.keys() if k <= 24])
vals_plot = [factors_cl[a] for a in ages_plot]

fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(ages_plot, vals_plot, 'o-', linewidth=2, markersize=6, color='steelblue')
ax.axhline(y=1.0, color='red', linestyle='--', linewidth=1.5, label='f = 1 (stable)')
ax.fill_between(ages_plot, 1.0, vals_plot, 
                 where=[v >= 1 for v in vals_plot],
                 alpha=0.2, color='green', label='Croissance')
ax.fill_between(ages_plot, vals_plot, 1.0,
                 where=[v < 1 for v in vals_plot],
                 alpha=0.2, color='red', label='Decroissance')
ax.set_xlabel('Age du pret (mois)', fontsize=12)
ax.set_ylabel('Facteur de developpement', fontsize=12)
ax.set_title('Facteurs Chain Ladder (contrats post-2021)', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0.8, 1.5)
plt.tight_layout()
plt.show()

## Etape 5 : Projection du triangle

Compléter les cohortes incomplètes en appliquant les facteurs.

In [ ]:
def project_triangle(triangle, factors, max_age=36):
    tri_proj = triangle.copy()
    
    for cohorte in tri_proj.index:
        last_valid = tri_proj.loc[cohorte].last_valid_index()
        
        if last_valid is None:
            continue
        
        current_val = tri_proj.loc[cohorte, last_valid]
        
        for age in tri_proj.columns:
            if age <= last_valid or age > max_age:
                continue
            
            prev_age = tri_proj.columns[list(tri_proj.columns).index(age) - 1]
            factor = factors.get(prev_age, 1.0)
            current_val = current_val * factor
            tri_proj.loc[cohorte, age] = current_val
    
    return tri_proj

triangle_projete = project_triangle(triangle_filtre, factors_cl, max_age=36)

print(f"Taux remplissage avant projection : {(~triangle_filtre.isna()).sum().sum() / triangle_filtre.size:.1%}")
print(f"Taux remplissage après projection : {(~triangle_projete.isna()).sum().sum() / triangle_projete.size:.1%}")

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

sns.heatmap(triangle_filtre.iloc[:, :24], annot=False, cmap='YlOrRd',
            ax=axes[0], vmin=0, vmax=1)
axes[0].set_title('Triangle OBSERVE', fontsize=13)
axes[0].set_xlabel('Age (mois)')
axes[0].set_ylabel('Cohorte')

sns.heatmap(triangle_projete.iloc[:, :24], annot=False, cmap='YlOrRd',
            ax=axes[1], vmin=0, vmax=1)
axes[1].set_title('Triangle PROJETE (Chain Ladder)', fontsize=13)
axes[1].set_xlabel('Age (mois)')
axes[1].set_ylabel('Cohorte')

plt.tight_layout()
plt.show()

## Etape 6 : Courbe ER baseline (depuis octroi)

$$ER\_baseline(t) = \text{moyenne}(ER\_projete(c, t)) \text{ sur toutes les cohortes}$$

C'est la **structure par terme depuis l'octroi**.

In [ ]:
# Nombre de cohortes par âge
nb_cohortes = triangle_projete.notna().sum(axis=0)

# Ne garder que les âges avec >= 3 cohortes
seuil_min = 3
ages_fiables = nb_cohortes[nb_cohortes >= seuil_min].index
er_baseline = triangle_projete[ages_fiables].mean(axis=0, skipna=True)

# Intervalles de confiance
er_q10 = triangle_projete[ages_fiables].quantile(0.10, axis=0)
er_q90 = triangle_projete[ages_fiables].quantile(0.90, axis=0)

print("COURBE ER BASELINE (DEPUIS OCTROI)")
print("=" * 70)
print(f"{'Age':>5} {'ER baseline':>12} {'Nb cohortes':>15}")
print("-" * 70)
for age in sorted(er_baseline.index):
    if not np.isnan(er_baseline[age]):
        print(f"{age:>5.0f} {er_baseline[age]:>12.4f} {nb_cohortes[age]:>15.0f}")

# Visualisation
fig, axes = plt.subplots(2, 1, figsize=(14, 12))

ages_plot = sorted(er_baseline.index)
vals_baseline = [er_baseline[a] for a in ages_plot]
vals_q10 = [er_q10.get(a, 0) for a in ages_plot]
vals_q90 = [er_q90.get(a, er_baseline[a]) for a in ages_plot]

# Courbe baseline
axes[0].plot(ages_plot, vals_baseline, linewidth=3, color='steelblue', 
             marker='o', markersize=5, label='ER baseline (moyenne)')
axes[0].fill_between(ages_plot, vals_q10, vals_q90,
                      alpha=0.2, color='steelblue', label='Intervalle Q10-Q90')
axes[0].set_xlabel('Age du pret depuis octroi (mois)', fontsize=12)
axes[0].set_ylabel('ER cumule baseline', fontsize=12)
axes[0].set_title('STRUCTURE PAR TERME DEPUIS OCTROI\n(Contrats post-2021 uniquement)', 
                  fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, max(vals_baseline) * 1.15)

# Increments
increments = er_baseline.diff().fillna(0)
ages_inc = sorted([a for a in increments.index if a > 0])
vals_inc = [increments[a] for a in ages_inc]

axes[1].bar(ages_inc, vals_inc, color='darkorange', edgecolor='black', alpha=0.7)
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=1)
axes[1].set_xlabel('Age du pret (mois)', fontsize=12)
axes[1].set_ylabel('Increment ER mensuel', fontsize=12)
axes[1].set_title('Vitesse d\'accumulation des RA', fontsize=13)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"\nER baseline a 12 mois : {er_baseline.get(12, np.nan):.2%}")
print(f"ER baseline a 24 mois : {er_baseline.get(24, np.nan):.2%}")
print(f"ER baseline max (age {int(ages_plot[-1])}) : {vals_baseline[-1]:.2%}")

## Etape 7 : Sauvegarde

Sauvegarder la courbe ER(t) depuis l'octroi pour comparaison.

In [ ]:
# Sauvegarder
er_baseline.to_csv('er_baseline_depuis_octroi_post2021.csv', header=['ER_baseline'])

# Facteurs
factors_df = pd.DataFrame({
    'Age': list(factors_cl.keys()),
    'Facteur': list(factors_cl.values())
})
factors_df.to_csv('cl_factors_post2021.csv', index=False)

# Triangle projeté
triangle_projete.to_csv('triangle_projete_post2021.csv')

print("FICHIERS SAUVEGARDES :")
print("  - er_baseline_depuis_octroi_post2021.csv")
print("  - cl_factors_post2021.csv")
print("  - triangle_projete_post2021.csv")

## Etape 8 : Comparaison avec approche horizon résiduel

Chargeons la courbe de l'approche principale pour comparer.

In [ ]:
# Charger la courbe par horizon résiduel (si existe)
try:
    er_horizon_residuel = pd.read_csv('table_h_ER.csv', index_col=0)
    er_horizon_residuel = er_horizon_residuel['ER_cumule']
    
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Courbe depuis octroi
    ages_octroi = sorted([a for a in er_baseline.index if a <= 24])
    vals_octroi = [er_baseline[a] for a in ages_octroi]
    ax.plot(ages_octroi, vals_octroi, 'o-', linewidth=3, markersize=7, 
            color='steelblue', label='Depuis octroi (post-2021)')
    
    # Courbe par horizon résiduel
    ages_horizon = sorted([a for a in er_horizon_residuel.index if a <= 24])
    vals_horizon = [er_horizon_residuel[a] for a in ages_horizon]
    ax.plot(ages_horizon, vals_horizon, 's--', linewidth=3, markersize=7,
            color='darkorange', label='Par horizon résiduel (tous contrats)', alpha=0.7)
    
    ax.set_xlabel('Horizon (mois ou echéances)', fontsize=12)
    ax.set_ylabel('ER cumule', fontsize=12)
    ax.set_title('COMPARAISON DES DEUX APPROCHES\nDepuis octroi vs Horizon résiduel', 
                 fontsize=14, fontweight='bold')
    ax.legend(fontsize=11, loc='best')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, max(max(vals_octroi), max(vals_horizon)) * 1.1)
    
    plt.tight_layout()
    plt.show()
    
    print("\nCOMPARAISON (à 12 mois) :")
    er_12_octroi = er_baseline.get(12, np.nan)
    er_12_horizon = er_horizon_residuel.get(12, np.nan)
    print(f"  Depuis octroi : {er_12_octroi:.2%}")
    print(f"  Horizon résiduel : {er_12_horizon:.2%}")
    print(f"  Écart : {abs(er_12_octroi - er_12_horizon):.2%}")
    
except FileNotFoundError:
    print("Fichier table_h_ER.csv introuvable")
    print("Exécutez d'abord le notebook avec l'approche par horizon résiduel")

## Synthèse

### Approche "depuis octroi" (ce notebook)

**Avantages** :
- ✅ Approche **classique en actuariat** (comparable à la littérature)
- ✅ Interprétation : "ER après t mois de vie du prêt"
- ✅ Pas de biais si on filtre post-2021
- ✅ Triangles Chain Ladder standards

**Limites** :
- ⚠️ **Moins de données** (seulement post-2021)
- ⚠️ Triangle plus **sparse** (cohortes récentes incomplètes)
- ⚠️ Peu d'âges observables (max 36 mois si octroi en 2021)

### Comparaison avec "horizon résiduel"

| Critère | Depuis octroi | Horizon résiduel |
|---------|---------------|------------------|
| **Données utilisées** | Post-2021 uniquement | Tous les contrats |
| **Taille échantillon** | ~10-30% | 100% |
| **Ages observables** | Max 36 mois | Jusqu'à 120+ mois |
| **Question** | "ER après t mois ?" | "Proba RA dans les h prochains mois ?" |
| **Opérationnel ALM** | ✓ | ✓✓ (meilleur) |
| **Académique** | ✓✓ (standard) | ✓ (innovant) |

### Recommandation

Pour votre PFE :
1. **Approche principale** : Horizon résiduel (plus de données, opérationnel)
2. **Approche secondaire** : Depuis octroi (ce notebook, pour comparaison académique)
3. **Mentionner** : Les deux donnent des résultats cohérents malgré les perspectives différentes

Cela montre votre maîtrise des deux méthodologies ! 🎯